In [1]:
# 중첩 객체와 배열이 포함된 이커머스 영수증 스키마를 정의합니다.
receipt_schema = {
    "title": "ReceiptData",
    "description": "영수증 텍스트에서 결제 정보와 구매 품목 목록을 추출하는 스키마",
    "type": "object",
    "properties": {
        "store_info": {
            # 중첩 객체: 매장 정보
            "type": "object",
            "properties": {
                "store_name": {"type": "string", "description": "상점 이름"},
                "branch": {"type": "string", "description": "지점명"}
            },
            "required": ["store_name"]
        },
        "items": {
            # 배열 구조: 구매 품목 리스트
            "type": "array",
            "description": "구매한 상품 항목들의 목록",
            "items": {
                "type": "object",
                "properties": {
                    "item_name": {"type": "string", "description": "상품명"},
                    "quantity": {"type": "integer", "description": "구매 수량"},
                    "price": {"type": "integer", "description": "개당 가격"}
                },
                "required": ["item_name", "quantity", "price"]
            }
        },
        "payment_type": {
            "type": "string",
            "enum": ["카드", "현금", "간편결제"],
            "description": "결제 수단"
        }
    },
    "required": ["store_info", "items", "payment_type"]
}

receipt_text = """
[스마트스토어 강남점] 영수증
- 무선 마우스 2개 (개당 25,000원)
- C타입 케이블 1개 (개당 12,000원)
신용카드로 일시불 결제 완료되었습니다.
"""

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="gpt-6-luna",
    model_provider="openai",
    reasoning_effort="low"
)

structured_model = model.with_structured_output(receipt_schema)

result = structured_model.invoke(receipt_text)

print(result)

{'store_info': {'store_name': '스마트스토어', 'branch': '강남점'}, 'items': [{'item_name': '무선 마우스', 'quantity': 2, 'price': 25000}, {'item_name': 'C타입 케이블', 'quantity': 1, 'price': 12000}], 'payment_type': '카드'}
